In [60]:
import re
import tiktoken
from chromadb import Client
from chromadb.config import Settings
from docling.document_converter import DocumentConverter

In [ ]:
def get_heading(chunk):
    match = re.match(r"^#{1,6}\s+(.*)", chunk.strip().splitlines()[0])
    return match.group(1).strip() if match else "Unknown"

enc = tiktoken.get_encoding("cl100k_base")


In [4]:
path = r"C:\\Users\\Zigron\\Documents\\research papers\\BLIP.pdf"
converter = DocumentConverter()

In [6]:
doc = converter.convert(path).document

2025-09-10 15:02:04,856 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-09-10 15:02:04,863 - INFO - Going to convert document batch...
2025-09-10 15:02:04,864 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 409740a308d273b3f090712d250cd783
2025-09-10 15:02:04,865 - INFO - Accelerator device: 'cpu'
2025-09-10 15:02:08,857 - INFO - Accelerator device: 'cpu'
2025-09-10 15:02:13,259 - INFO - Accelerator device: 'cpu'
2025-09-10 15:02:15,262 - INFO - Loading plugin 'docling_defaults'
2025-09-10 15:02:15,262 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-09-10 15:02:15,278 - INFO - Processing document BLIP.pdf
c:\Users\Zigron\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
2025-09-10 15:04:39,061 - INFO - Finished converting document BLIP.pdf in 154.20 

In [8]:
md = doc.export_to_markdown()

In [55]:
import re
import tiktoken

# ---- Tokenizer ----
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def split_long_chunk(text, section, position, max_tokens=800, overlap=100):
    tokens = enc.encode(text)
    chunks = []
    start = 0
    sub_pos = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)

        chunks.append({
            "text": chunk_text,
            "metadata": {
                "section": section,
                "position": f"{position}.{sub_pos}"  # keep sub-index
            }
        })

        start += max_tokens - overlap
        sub_pos += 1

    return chunks

# ---- Chunking Pipeline ----
def chunk_markdown(md: str, max_tokens=800, overlap=100):
    # Find all heading positions
    positions = [m.start() for m in re.finditer(r"^#{1,6}\s", md, re.MULTILINE)]
    positions = positions[1:]

    chunks = []
    for i in range(len(positions)-1):
        chunks.append(md[positions[i]:positions[i+1]])
    chunks.append(md[positions[-1]:])  # last section

    final_chunks = []
    for idx, chunk in enumerate(chunks):
        # Extract section heading
        first_line = chunk.strip().splitlines()[0]
        heading_match = re.match(r"^#{1,6}\s+(.*)", first_line)
        section = heading_match.group(1).strip() if heading_match else "Unknown"

        if count_tokens(chunk) > max_tokens:
            final_chunks.extend(split_long_chunk(chunk, section, idx, max_tokens, overlap))
        else:
            final_chunks.append({
                "text": chunk,
                "metadata": {
                    "section": section,
                    "position": str(idx)
                }
            })

    return final_chunks


In [56]:
chunks = chunk_markdown(md)

    # Show first few chunks
for ch in chunks[:5]:
    print(ch["metadata"])
    print(ch["text"][:200], "...\n")


{'section': 'Abstract', 'position': '0'}
## Abstract

Vision-Language Pre-training (VLP) has advanced the performance for many vision-language tasks. However, most existing pre-trained models only excel in either understanding-based tasks or ...

{'section': '1. Introduction', 'position': '1'}
## 1. Introduction

Vision-language pre-training has recently received tremendous success on various multimodal downstream tasks. However, existing methods have two major limitations:

- (1) Model per ...

{'section': '2. Related Work', 'position': '2'}
## 2. Related Work

 ...

{'section': '2.1. Vision-language Pre-training', 'position': '3'}
## 2.1. Vision-language Pre-training

Vision-language pre-training (VLP) aims to improve performance of downstream vision and language tasks by pretraining the model on large-scale image-text pairs. D ...

{'section': '2.2. Knowledge Distillation', 'position': '4'}
## 2.2. Knowledge Distillation

Knowledge distillation (KD) (Hinton et al., 2015) aims to imp

In [ ]:
client = Client(Settings(persist_directory="./chroma_store"))
collection = client.get_or_create_collection("papers")

